<a href="https://colab.research.google.com/github/joanby/tensorflow2/blob/master/Collab%208%20-%20Reinforcement%20Learning%20para%20problemas%20de%20Stock%20Market%20Trading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Paso 1: Instalar las dependencias y configurar el entorno

In [1]:
!pip install -U "tensorflow==2.20.0"

In [2]:
!pip install -U yfinance

In [3]:
!pip install pandas-datareader

## Paso 2: Importar las dependencias del proyecto

In [4]:
import yfinance as yf
import math
import random
import numpy as np
import pandas as pd
import tensorflow as tf

import matplotlib.pyplot as plt
import pandas_datareader as data_reader

from tqdm import tqdm_notebook, tqdm
from collections import deque
from datetime import datetime

In [5]:
tf.__version__

'2.20.0'

## Paso 3: Construir la red neuronal de la AI del Trader

In [6]:
# Esta clase sera un boot
class AI_Trader():
  # Creamos el constructor con el nombre del modelo como AITrader
  def __init__(self, state_size, action_space=3, model_name="AITrader"): #Manten, Compra, Vende

    # Tamaño de los estados
    self.state_size = state_size
    # Numero de accion que va a poder tomar
    self.action_space = action_space
    # definimos la momoria de longitud maxima deque 2.000
    self.memory = deque(maxlen=2000)
    # Inventario es una lista vacia que guardaremos las divisas
    self.inventory = []
    self.model_name = model_name

    # Definimos para la fase de aprendizaje:
    self.gamma = 0.95
    self.epsilon = 1.0
    self.epsilon_final = 0.01
    self.epsilon_decay = 0.995

    # Inicializamos la funcion que crearemos a continuacion
    self.model = self.model_builder()

  # Esta sera la funcion de crear el modelo
  def model_builder(self):
    # Definimos el modelo a trabajar utilizando la clase Sequential
    model = tf.keras.models.Sequential()

    # Ahora la capa de entrada indicando el numero de estados que tendra
    model.add(tf.keras.layers.Dense(units=32, activation='relu', input_dim=self.state_size))

    # Creamos mas capas ocultas duplicando las neuronas
    model.add(tf.keras.layers.Dense(units=64, activation='relu'))
    model.add(tf.keras.layers.Dense(units=128, activation='relu'))

    # Esta es la capa de salida con el espacio de salida con la funcion de activacion lineal
    model.add(tf.keras.layers.Dense(units=self.action_space, activation='linear'))

    # Ahora compilamos el modelo, con la funcion de perdida media, con el optimizador de adam
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate=0.001))

    # Devolvemos el modelo
    return model

  # Funcion de trade para la toma de deciciones de comprar, vender o mantener
  def trade(self, state):
    # SI al generar un numero aleatorio, el numero sale mas pequeño que el
    if random.random() <= self.epsilon:
      # devolvemos un numero aleatorio entre 0 y 2 incluyendo ambos
      return random.randrange(self.action_space)

    # En caso contrario definimos el confunto de acciones de acuerdo a la prediccion de acuerdo al estado actual
    actions = self.model.predict(state)
    # Devolvemos la accion con mayor probabilidad
    return np.argmax(actions[0])

  # Funcion para entrenar con el argumento del tamaño del bloque a entrenar
  def batch_train(self, batch_size):

    # Creamos una lista vacia
    batch = []
    # Iteramos dentro del bloque en memoria - batch_size de la longitud de la memoria (eteramos de atras hacia adelante)
    for i in range(len(self.memory) - batch_size + 1, len(self.memory)):
      # Apendizamos los ultimos bloques de memoria
      batch.append(self.memory[i])

    # Cada uno de los bloques que hay en la memoria contara con un estado, una accion, la recompenza, el estado siguiente y saber si finalizo el ejercicio
    for state, action, reward, next_state, done in batch:
      # definimos la variable reward
      reward = reward
      # Si no hemos terminado
      if not done:
        # actualizamos la recompenza
        reward = reward + self.gamma * np.amax(self.model.predict(next_state)[0])

      # El targe sera el que me predice el modelo a partir de un estado
      target = self.model.predict(state)
      # Para la accion actual que toma el agente, sera la recompenza que hemos calculado recientemente
      target[0][action] = reward

      # Le indicamos al modelo que propague el error hacia atras, pasandole el estado actual, el target, una epoca y verbose igual a 0 para evitar que imprima
      self.model.fit(state, target, epochs=1, verbose=0)

    # Decrementamos
    if self.epsilon > self.epsilon_final:
      self.epsilon *= self.epsilon_decay

## Paso 4: Pre procesado del dataset

### Definir las funciones adicionales

#### Sigmoide

In [7]:
def sigmoid(x):
  return 1 / (1 + math.exp(-x))

#### Función de formato de precios

In [8]:
def stocks_price_format(n):
  if n < 0:
    return "- $ {0:2f}".format(abs(n))
  else:
    return "$ {0:2f}".format(abs(n))

#### Carga del dataset

In [9]:
def dataset_loader(stock_name):
    inicio = datetime(2010, 1, 4)
    fin = datetime(2019, 8, 3)

    dataset = yf.download([stock_name], start=inicio, end=fin)

    start_date = str(dataset.index[0]).split()[0]
    end_date = str(dataset.index[-1]).split()[0]

    close = dataset['Close']

    return close

### State creator

In [10]:
def state_creator(data, timestep, window_size):
  starting_id = timestep - window_size + 1

  if starting_id >= 0:
    # Obtener valores planos como lista de floats
    # .values (NumPy array) -> .flatten() (array 1D) -> .tolist()
    windowed_data = data.iloc[starting_id:timestep+1].values.flatten().tolist()
  else:
    # 1. Obtener el valor escalar FLOAT para relleno
    filler = data.iloc[0].item()

    # 2. Obtener la porción de datos como lista plana de floats
    window_slice = data.iloc[0:timestep+1].values.flatten().tolist()

    # Concatenación de listas de floats puros
    windowed_data = - starting_id * [filler] + window_slice

  state = []
  for i in range(window_size - 1):
    # La resta ahora será (float - float)
    state.append(sigmoid(windowed_data[i+1] - windowed_data[i]))

  return np.array([state])

### Cargar una divisa de mercado

In [11]:
stock_name = "AAPL"
data = dataset_loader(stock_name)

/tmp/ipython-input-2612579391.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  dataset = yf.download([stock_name], start=inicio, end=fin)
[*********************100%***********************]  1 of 1 completed


In [12]:
data.head()

Ticker,AAPL
Date,
2010-01-04,6.418382
2010-01-05,6.429480
2010-01-06,6.327211
2010-01-07,6.315514
2010-01-08,6.357503


## Paso 5: Entrenar la AI Trader

### Configurar los hyper parámetros

In [13]:
# Definiremos el tamaño de la ventana (10 dias)
window_size = 10
# Seran 1.000 epocas
episodes = 1000

# Para agilizar crearemos una muestra de 32 datos
batch_size = 32
# La muestra son todos los datos menos el ultimo
data_samples = len(data) - 1

### Definir el modelo del AI Trader

In [14]:
# Definiremos un objeto de la clase AI_Trader, pasandole los 10 dias del pasado
trader = AI_Trader(window_size)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
trader.model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,171 (43.64 KB)

 Trainable params: 11,171 (43.64 KB)

 Non-trainable params: 0 (0.00 B)

### Bucle de entrenamiento

In [16]:
# Crearemos un bucle que entrene para cada uno de los episodios +1
for episode in range(1, episodes + 1):
  # Imprimimos el episodio en el que vamos
  print("Episodio: {}/{}".format(episode, episodes))

  # definimos el estado inicial, tomando los datos del dia 0 hasta el dia en que queremos predecir
  state = state_creator(data, 0, window_size + 1)

  # Creamos dos variables, una para el veneficio total y una lista para el inventario
  total_profit = 0
  trader.inventory = []

  # Para cada episodio tomaremos todos los dias hasta el penultimo
  for t in tqdm(range(data_samples)):
    # Las acciones seran aleatoreas
    action = trader.trade(state)

    # definimos el siguiente estado, con el dia siguiente
    next_state = state_creator(data, t+1, window_size + 1)
    # definimos la recompenza
    reward = 0

    # Almacenamos el precio de cierre del día actual como un escalar float
    current_price = data.iloc[t].item()

    # Si la accion es 1, solo debemos comprar
    if action == 1: #Compra
      # Lo agregamos al inventario como un escalar (float)
      trader.inventory.append(current_price)
      print("AI Trader compró: ", stocks_price_format(current_price))

    # Pero si la accion es 2 procedemos a vender si es que el inventario no esta vacio
    elif action == 2 and len(trader.inventory) > 0: #Vende
      # En este caso lo sacamos del inventario (buy_price ya es un float)
      buy_price = trader.inventory.pop(0)

      # La recompenza sera el valor maximo para el dia que estamos comprando, menos el precio de compra
      reward = max(current_price - buy_price, 0)
      # Veremos si la tranferencia es positiva o negativa
      total_profit += current_price - buy_price
      print("AI Trader vendió: ", stocks_price_format(current_price), " Beneficio: " + stocks_price_format(current_price - buy_price) )

    # Si el t es el ultimo, quiere decir que es la ultima fecha a analizar
    if t == data_samples - 1:
      done = True
    else:
      done = False

    # Guardamos todos los datos en la memoria
    trader.memory.append((state, action, reward, next_state, done))

    # Pasamos al siguiente estado para que la red neuronal siga trabajando
    state = next_state

    # Si acabamos todas las fechas...
    if done:
      print("########################")
      print("BENEFICIO TOTAL: {}".format(total_profit))
      print("########################")

    # Le indicamos que si tenemos mas informacion que memoria, actualizamos los pesos
    if len(trader.memory) > batch_size:
      trader.batch_train(batch_size)

  # Por cada 10 episodio guardando un fichero ai_trader_ con el numero de episodio .h5
  if episode % 10 == 0:
    trader.model.save("ai_trader_{}.h5".format(episode))


Episodio: 1/1000


  0%|          | 0/2411 [00:00<?, ?it/s]

AI Trader compró:  $ 6.327211
AI Trader compró:  $ 6.315514
AI Trader compró:  $ 6.357503
AI Trader compró:  $ 6.301420
AI Trader vendió:  $ 6.317612  Beneficio: - $ 0.009600
AI Trader vendió:  $ 6.281025  Beneficio: - $ 0.034488
AI Trader vendió:  $ 6.176056  Beneficio: - $ 0.181447
AI Trader compró:  $ 6.350004
AI Trader vendió:  $ 6.090280  Beneficio: - $ 0.211140
AI Trader compró:  $ 5.976917
AI Trader vendió:  $ 5.760080  Beneficio: - $ 0.589924
AI Trader compró:  $ 5.840156
AI Trader vendió:  $ 5.874048  Beneficio: - $ 0.102869
AI Trader vendió:  $ 5.975115  Beneficio: $ 0.134959
AI Trader compró:  $ 5.851851
AI Trader compró:  $ 5.958321
AI Trader compró:  $ 6.009606
AI Trader vendió:  $ 6.074686  Beneficio: $ 0.222835
AI Trader compró:  $ 6.086083
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 392ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━

  1%|▏         | 33/2411 [00:11<13:21,  2.97it/s]

AI Trader vendió:  $ 6.010803  Beneficio: $ 0.052483
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/st

...

AI Trader compró:  $ 7.740388
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━

  5%|▌         | 129/2411 [11:20<4:40:37,  7.38s/it]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━

  5%|▌         | 129/2411 [11:24<3:21:52,  5.31s/it]


KeyboardInterrupt: 

* Detuve la ejecucion ya que tardaria aproximadamete unas 5 horas